# Base de datos de empleos de analista de datos en el Perú

**Fuente:** [bumeran.com.pe](https://www.bumeran.com.pe) · **Herramienta:** Selenium

> **¿Qué ofertas de analista de datos hay hoy en el Perú, en qué condiciones se
> ofrecen y qué les piden a los postulantes?**

El resultado es una tabla con **una fila por aviso** y más de 70 columnas: puesto,
empresa, área, nivel, modalidad, contrato, vacantes, ubicación, fechas, la
descripción completa, la ficha y las calificaciones de la empresa, y las
herramientas que pide cada aviso (SQL, Python, Power BI…).

| Etapa | Qué hace | Resultado |
|---|---|---|
| 1. Listado | recorre las búsquedas y junta las direcciones de los avisos | `avisos` |
| 2. Detalle | entra a cada aviso y extrae todo lo que publica | `detalles` |
| 3. Limpieza | fechas, números y variables nuevas a partir del texto | `empleos` |
| 4. Guardar | CSV + Excel con diccionario de variables | `bumeran_analista_datos_<fecha>` |
| 5. Primeras respuestas | modalidad, nivel, ubicación, herramientas, experiencia | tablas |

Correrlo completo toma **cerca de media hora**: son unas 300 páginas, con una
pausa entre cada una.

## 0. Preparación

In [1]:
import json
import math
import re
import time
from datetime import datetime

import pandas as pd
from selenium import webdriver
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# La tabla final es ancha: que se vean todas las columnas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

### Configuración

**Qué se busca.** "Analista de datos" no es un solo nombre de puesto: el mismo
trabajo se publica como *Analista BI*, *Data Analyst* o *Analista de
Información*. Se buscan todas las variantes y luego se juntan.

Se agregan también cuatro búsquedas de **puestos afines** (ciencia e
ingeniería de datos). No son el mismo perfil, pero piden herramientas parecidas
y sirven de comparación. En la tabla quedan marcados en la columna `familia`,
así que se pueden filtrar.

**Los términos van sin tilde.** Al probar, Bumeran devolvió 0 resultados para
*analista de información* y 23 para *analista de informacion*.

In [2]:
TERMINOS_ANALISTA = [
    "analista de datos",
    "data analyst",
    "analista bi",
    "business intelligence",
    "inteligencia de negocios",
    "analista de informacion",
    "analista de reportes",
    "power bi",
    "data analytics",
    "analitica de datos",
]

TERMINOS_AFINES = [
    "ciencia de datos",
    "data scientist",
    "ingeniero de datos",
    "data engineer",
]

PAUSA_SEGUNDOS = 2       # entre página y página: para no saturar el servidor
HEADLESS = True          # False = ver la ventana de Chrome mientras trabaja
LIMITE_AVISOS = None     # un número (ej. 10) para una prueba rápida; None = todos

FECHA_EXTRACCION = datetime.now()
ARCHIVO_CSV = f"bumeran_analista_datos_{FECHA_EXTRACCION:%Y-%m-%d}.csv"
ARCHIVO_EXCEL = f"bumeran_analista_datos_{FECHA_EXTRACCION:%Y-%m-%d}.xlsx"

print(len(TERMINOS_ANALISTA) + len(TERMINOS_AFINES), "búsquedas ·", ARCHIVO_CSV)

14 búsquedas · bumeran_analista_datos_2026-09-23.csv


## 1. Abrir el navegador

Bumeran dibuja las ofertas con JavaScript: si se descarga la página sin
navegador, llega vacía. Por eso se usa Selenium, que controla un Chrome de
verdad.

Un detalle que se descubrió al probar: Chrome invisible (`--headless=new`) se
presenta ante el sitio como **`HeadlessChrome`**, y Bumeran, que está detrás de
Cloudflare, responde a eso con *"Sorry, you have been blocked"*. Por eso se le
indica que se presente como un Chrome normal, que es lo que realmente es, solo
que sin ventana.

In [3]:
opciones = Options()
if HEADLESS:
    opciones.add_argument("--headless=new")
opciones.add_argument("--window-size=1920,1080")
opciones.add_argument(
    "user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/140.0 Safari/537.36"
)

navegador = webdriver.Chrome(options=opciones)
print("Navegador abierto")

Navegador abierto


### Antes de extraer: el `robots.txt`

Es el archivo donde cada sitio indica qué rutas no quiere que se recorran
automáticamente. Se lee **antes** de tocar nada.

In [4]:
navegador.get("https://www.bumeran.com.pe/robots.txt")
print(navegador.find_element(By.TAG_NAME, "body").text)

User-agent: *
Disallow: /*recientes=true
Disallow: /*relevantes=true
Disallow: /empleos.html/111
Disallow: /empleos.html/100
Disallow: /empleos-area-*/111
Disallow: /empleos-area-*/100
Disallow: /en-*/empleos.html/111
Disallow: /en-*/empleos.html/100
Disallow: /empleos-busquedaext-
Disallow: /empleos/aptitus/*
Disallow: /*?localidades=*
Disallow: /*&localidades=*

Sitemap: https://www.bumeran.com.pe/sitemap_avisos_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_core_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_empresas_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_listados_ubicacion_bum.xml
Sitemap: https://www.bumeran.com.pe/sitemap_tags_bum.xml


Las dos rutas que se usan aquí, las búsquedas (`/empleos-busqueda-...`) y los
avisos (`/empleos/...html`), **no están prohibidas**. Sí se evitan las que el
sitio pide no tocar: los filtros `?localidades=`, `recientes=true` y
`relevantes=true`.

## 2. Etapa 1 · El listado de avisos

Bumeran pone la búsqueda en la propia dirección, así que no hace falta
escribir en el buscador:

```
https://www.bumeran.com.pe/empleos-busqueda-analista-de-datos.html?page=2
```

In [5]:
def url_busqueda(termino, pagina):
    # Bumeran une las palabras del término con guiones
    termino_en_url = termino.strip().lower().replace(" ", "-")
    return f"https://www.bumeran.com.pe/empleos-busqueda-{termino_en_url}.html?page={pagina}"


url_busqueda("analista de datos", 1)

'https://www.bumeran.com.pe/empleos-busqueda-analista-de-datos.html?page=1'

In [6]:
navegador.get(url_busqueda("analista de datos", 1))

# El título con el total aparece recién cuando llegaron los resultados
WebDriverWait(navegador, 20).until(
    EC.text_to_be_present_in_element((By.TAG_NAME, "h1"), "empleos")
)
time.sleep(PAUSA_SEGUNDOS)

navegador.find_element(By.TAG_NAME, "h1").text

'84 empleos para la búsqueda "analista de datos"'

Cada aviso es un enlace a `/empleos/<nombre-del-puesto>-<número>.html`. Se
buscan los enlaces **por ese patrón de dirección** y no por su clase CSS: las
clases de este sitio (`sc-ekQYnd bAylLX`) las genera un programa y cambian
cada vez que Bumeran actualiza su web.

In [7]:
avisos_en_pagina = []
for enlace in navegador.find_elements(By.CSS_SELECTOR, "a[href*='/empleos/']"):
    direccion = enlace.get_attribute("href")
    if re.search(r"-\d+\.html$", direccion):
        avisos_en_pagina.append(direccion)

print(len(avisos_en_pagina), "avisos en la página 1")
avisos_en_pagina[:3]

20 avisos en la página 1


['https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html',
 'https://www.bumeran.com.pe/empleos/analista-de-datos-1118440330.html',
 'https://www.bumeran.com.pe/empleos/analista-de-datos-operativos-oslo-1118453111.html']

Bumeran muestra 20 avisos por página y el total aparece en el título. Con eso
se sabe cuántas páginas hay que recorrer para cada búsqueda.

In [8]:
def leer_listado(navegador, termino):
    navegador.get(url_busqueda(termino, 1))
    WebDriverWait(navegador, 20).until(
        EC.text_to_be_present_in_element((By.TAG_NAME, "h1"), "empleos")
    )
    time.sleep(PAUSA_SEGUNDOS)

    titulo = navegador.find_element(By.TAG_NAME, "h1").text
    total = int(titulo.split()[0].replace(".", ""))   # "84 empleos para..." -> 84
    paginas = math.ceil(total / 20)

    encontrados = []
    for pagina in range(1, paginas + 1):
        if pagina > 1:
            navegador.get(url_busqueda(termino, pagina))
            WebDriverWait(navegador, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/empleos/']"))
            )
            time.sleep(PAUSA_SEGUNDOS)

        for enlace in navegador.find_elements(By.CSS_SELECTOR, "a[href*='/empleos/']"):
            direccion = enlace.get_attribute("href")
            if re.search(r"-\d+\.html$", direccion):
                encontrados.append({"url": direccion, "termino": termino, "pagina": pagina})

    distintos = len(set(fila["url"] for fila in encontrados))
    print(f"  {termino:<26} {total:>4} anunciados · {distintos:>4} leídos")
    return encontrados

In [9]:
inicio = time.time()
resultados_listado = []
for termino in TERMINOS_ANALISTA + TERMINOS_AFINES:
    resultados_listado.extend(leer_listado(navegador, termino))

listado = pd.DataFrame(resultados_listado)
print(f"\n{len(listado)} resultados en {(time.time() - inicio) / 60:.1f} minutos")

  analista de datos            84 anunciados ·   84 leídos


  data analyst                 18 anunciados ·   18 leídos


  analista bi                  30 anunciados ·   30 leídos


  business intelligence        27 anunciados ·   27 leídos


  inteligencia de negocios     12 anunciados ·   12 leídos


...

  analitica de datos            5 anunciados ·    5 leídos


  ciencia de datos              5 anunciados ·    5 leídos


  data scientist               10 anunciados ·   10 leídos


  ingeniero de datos           34 anunciados ·   34 leídos


  data engineer                41 anunciados ·   41 leídos

316 resultados en 1.2 minutos


Un mismo aviso aparece en varias búsquedas: un "Analista BI" sale al buscar
*analista bi*, *power bi* y *business intelligence*. Se deja **una fila por
aviso** y se anota en qué búsquedas apareció.

In [10]:
# Si el orden de los resultados cambia entre una página y otra, un aviso
# puede repetirse dentro de la misma búsqueda
listado = listado.drop_duplicates(subset=["url", "termino"])

avisos = listado.groupby("url", as_index=False).agg(
    terminos_busqueda=("termino", "; ".join),
    n_busquedas=("termino", "count"),
)

print("Resultados:", len(listado), "→ avisos distintos:", len(avisos))
avisos["n_busquedas"].value_counts().sort_index()

Resultados: 316 → avisos distintos: 288


n_busquedas
1    261
2     26
3      1
Name: count, dtype: int64

## 3. Etapa 2 · Entrar a cada aviso

La tarjeta del listado trae poco. El detalle del aviso lo trae todo, en tres
formatos distintos, y cada uno tiene una forma estable de ubicarse:

| Dónde está | Cómo se ubica | Qué trae |
|---|---|---|
| Bloque **JSON-LD** | `<script type="application/ld+json">` | puesto, empresa, fechas, ubicación, tipo de empleo, sueldo |
| **Lista de atributos** | el nombre del ícono: `icon-light-clock`, `icon-light-award`… | modalidad, subárea, jornada, nivel, vacantes, apto discapacidad |
| **Textos con etiqueta** | la etiqueta visible: «Industria», «Tamaño de la empresa»… | ficha y calificaciones de la empresa |

Ninguna depende de clases CSS generadas. Primero se revisa un aviso a mano.

In [11]:
url_ejemplo = listado.loc[listado["termino"] == "analista de datos", "url"].iloc[0]

navegador.get(url_ejemplo)
WebDriverWait(navegador, 20).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "ul[aria-label='Información adicional del aviso']"))
)
time.sleep(1.5)

print(url_ejemplo)
navegador.find_element(By.TAG_NAME, "h1").text

https://www.bumeran.com.pe/empleos/analista-de-datos-fractal-soluciones-it-1118447809.html


'Analista de Datos'

### 3.1 El bloque JSON-LD

Es un resumen del aviso en un formato estándar
([schema.org/JobPosting](https://schema.org/JobPosting)) que el sitio publica
para que Google muestre sus empleos en los resultados de búsqueda. Como está
hecho para que lo lean programas, es lo más estable de toda la página.

In [12]:
def leer_json_ld(navegador):
    # Una página puede traer varios bloques; se guardan según su tipo
    bloques = {}
    for script in navegador.find_elements(By.CSS_SELECTOR, "script[type='application/ld+json']"):
        contenido = json.loads(script.get_attribute("innerHTML"))
        if isinstance(contenido, dict) and "@type" in contenido:
            bloques[contenido["@type"]] = contenido
    return bloques


bloques = leer_json_ld(navegador)
list(bloques)

['JobPosting', 'BreadcrumbList', 'LocalBusiness']

In [13]:
for clave, valor in bloques["JobPosting"].items():
    if clave != "description":          # la descripción es larga: se ve más abajo
        print(f"{clave:<20} {valor}")

title                Analista de Datos
datePosted           2026-09-18T05:00:00.000Z
validThrough         2026-12-17T05:00:00.000Z
employmentType       TEMPORARY
jobLocation          {'address': {'addressLocality': 'Lima', 'addressRegion': 'Lima', '@type': 'PostalAddress', 'addressCountry': 'PE'}, '@type': 'Place'}
hiringOrganization   {'@type': 'Organization', 'name': 'FRACTAL SOLUCIONES IT ', 'sameAs': 'https://www.bumeran.com.pe/perfiles/empresa_fractal-soluciones-it_10199506.html', 'logo': 'https://imgbum.jobscdn.com/portal/img/empresas/11/static/logoMainPic_10199506_bum_vcc4f1e1d.jpg'}
baseSalary           {'@type': 'MonetaryAmount', 'currency': 'PEN', 'value': {'@type': 'QuantitativeValue', 'value': '', 'unitText': 'MONTH'}}
@type                JobPosting
@context             http://schema.org


Dos avisos a tener en cuenta al usar este bloque:

- El sueldo (`baseSalary`) viene **siempre vacío**, incluso en los avisos que
  sí muestran sueldo en pantalla. El sueldo se lee de la lista de atributos.
- Al recorrer todos los avisos aparecieron algunos **sin bloque `JobPosting`**.
  Para ellos, la empresa se toma del bloque `LocalBusiness` y la fecha, del
  texto visible «Publicado el 14/09/2026».

### 3.2 La lista de atributos

Cada dato va acompañado de un ícono, y **el ícono tiene nombre**. Así se sabe
qué es cada línea sin depender de su posición, que cambia: «Apto personas con
discapacidad» y el sueldo (`icon-light-money`, «De S/.1,800 A S/.2,100 Por
Mes») aparecen solo en algunos avisos y, cuando aparecen, corren a los demás.

In [14]:
for item in navegador.find_elements(By.CSS_SELECTOR, "ul[aria-label='Información adicional del aviso'] li"):
    icono = item.find_element(By.TAG_NAME, "i").get_attribute("name")
    print(f"{icono:<26} →  {item.text}")

icon-light-office          →  Híbrido
icon-light-cube            →  Tecnologia / Sistemas
icon-light-clock           →  Full-time, Temporal
icon-light-award           →  Semi Sr
icon-light-people          →  2 vacantes disponibles


### 3.3 La ficha de la empresa

Aquí no hay ícono ni JSON. Hay una etiqueta visible («Industria») y, justo
después, el valor. XPath permite pedir exactamente eso: *el elemento que sigue
al párrafo que dice «Industria»*.

In [15]:
def buscar_texto(navegador, xpath):
    # Si el elemento no existe (empresa confidencial, dato no publicado) devuelve None
    elementos = navegador.find_elements(By.XPATH, xpath)
    if elementos:
        return elementos[0].text.strip()
    return None


for etiqueta in ["Industria", "Ubicación", "Tamaño de la empresa"]:
    valor = buscar_texto(navegador, f"//p[normalize-space()='{etiqueta}']/following-sibling::*[1]")
    print(f"{etiqueta:<22} {valor}")

Industria              Informática / Tecnología
Ubicación              LINCE, Lima
Tamaño de la empresa   Entre 151 y 300 empleados


### 3.4 Todo junto en una función

Esto se repite para cada aviso, así que va en una función. Dos piezas más:

- **Calificaciones de la empresa.** Vienen dentro del `aria-label`, el texto
  que leen los lectores de pantalla para personas ciegas: `"Salario 3.7 estrellas"`.
- **Marcas sueltas** como «Nuevo» o «Postulación rápida» aparecen como líneas
  arriba de la descripción.

Todas las columnas arrancan en `None`, para que un aviso sin ese dato
igual tenga la columna.

In [16]:
# Qué dato es cada ícono de la lista de atributos
ICONOS = {
    "icon-light-office": "modalidad",
    "icon-light-cube": "subarea",
    "icon-light-clock": "jornada_y_contrato",
    "icon-light-award": "nivel_laboral",
    "icon-light-people": "vacantes_texto",
    "icon-light-discapacity": "apto_discapacidad_texto",
    "icon-light-money": "salario_texto",
}

# Inicio del aria-label de cada calificación de la empresa
CALIFICACIONES = {
    "calif_general": "Calificación general",
    "calif_ambiente_laboral": "Ambiente laboral",
    "calif_balance_vida_trabajo": "Balance laboral-personal",
    "calif_beneficios": "Beneficios",
    "calif_desarrollo_profesional": "Desarrollo profesional",
    "calif_gestion_liderazgo": "Gestión y liderazgo",
    "calif_salario": "Salario",
}

LISTA_ATRIBUTOS = "ul[aria-label='Información adicional del aviso']"

In [17]:
def leer_aviso(navegador, url):
    navegador.get(url)
    WebDriverWait(navegador, 20).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, LISTA_ATRIBUTOS))
    )
    time.sleep(1.5)     # las calificaciones de la empresa llegan un instante después

    aviso = {"url": url, "antiguedad_texto": None, "area": None,
             "personas_quieren_trabajar_aqui": None}
    for columna in ICONOS.values():
        aviso[columna] = None
    for columna in CALIFICACIONES:
        aviso[columna] = None

    # 1) JSON-LD
    bloques = leer_json_ld(navegador)
    oferta = bloques.get("JobPosting", {})
    organizacion = oferta.get("hiringOrganization", {})
    negocio = bloques.get("LocalBusiness", {})       # respaldo si falta JobPosting
    direccion = oferta.get("jobLocation", {}).get("address", {})

    aviso["puesto"] = oferta.get("title") or buscar_texto(navegador, "//h1")
    aviso["empresa"] = (organizacion.get("name") or negocio.get("name") or "").strip()
    aviso["empresa_url"] = organizacion.get("sameAs") or negocio.get("url")
    aviso["empresa_id"] = negocio.get("@id")
    aviso["fecha_publicacion_iso"] = oferta.get("datePosted")
    aviso["fecha_vencimiento_iso"] = oferta.get("validThrough")
    aviso["tipo_empleo_schema"] = oferta.get("employmentType")
    aviso["departamento"] = direccion.get("addressRegion")
    aviso["distrito"] = direccion.get("addressLocality")
    aviso["pais"] = direccion.get("addressCountry")

    # 2) Lista de atributos: el nombre del ícono dice qué dato es
    for item in navegador.find_elements(By.CSS_SELECTOR, LISTA_ATRIBUTOS + " li"):
        iconos = item.find_elements(By.TAG_NAME, "i")
        if iconos:
            nombre_icono = iconos[0].get_attribute("name")
            columna = ICONOS.get(nombre_icono, "otro_" + nombre_icono)
            aviso[columna] = item.text.strip()

    # 3) Textos ubicados por su etiqueta
    aviso["ubicacion_texto"] = buscar_texto(navegador, "//i[@name='icon-light-location-pin']/following-sibling::h2")
    aviso["descripcion"] = buscar_texto(navegador, "//*[@id='descripcion-aviso']")
    # Algunas descripciones traen su propio «Publicado el: ...»: se busca fuera de ellas
    aviso["publicado_el_texto"] = buscar_texto(
        navegador,
        "//p[starts-with(normalize-space(), 'Publicado el ') and not(ancestor::*[@id='descripcion-aviso'])]",
    )
    aviso["empresa_industria"] = buscar_texto(navegador, "//p[normalize-space()='Industria']/following-sibling::*[1]")
    aviso["empresa_ubicacion"] = buscar_texto(navegador, "//p[normalize-space()='Ubicación']/following-sibling::*[1]")
    aviso["empresa_tamano"] = buscar_texto(navegador, "//p[normalize-space()='Tamaño de la empresa']/following-sibling::*[1]")
    aviso["empresa_descripcion"] = buscar_texto(navegador, "//*[@id='descriptionCompany']")

    # El área es el primer enlace del breadcrumb que apunta a /empleos-area-... sin subárea
    for enlace in navegador.find_elements(By.CSS_SELECTOR, "nav[aria-label='Breadcrumb'] a"):
        destino = enlace.get_attribute("href") or ""
        if "/empleos-area-" in destino and "-subarea-" not in destino:
            aviso["area"] = enlace.text.strip()
            break

    # 4) Marcas: líneas sueltas en la cabecera y arriba de la descripción
    cabecera = navegador.find_element(By.ID, "header-component")
    lineas_cabecera = cabecera.text.split("\n")
    texto_detalle = navegador.find_element(By.ID, "section-detalle").text
    lineas_arriba = texto_detalle.split("Descripción del puesto")[0].split("\n")

    aviso["empresa_verificada"] = len(cabecera.find_elements(By.CSS_SELECTOR, "[aria-label='Empresa verificada']")) > 0
    aviso["alta_revision_perfiles"] = "Alta revisión de perfiles" in lineas_cabecera
    aviso["aviso_nuevo"] = "Nuevo" in lineas_arriba
    aviso["multiples_vacantes"] = "Múltiples vacantes" in lineas_arriba
    aviso["postulacion_rapida"] = "Postulación rápida" in lineas_arriba
    for linea in lineas_arriba:
        if linea.startswith("Publicado") or linea.startswith("Actualizado"):
            aviso["antiguedad_texto"] = linea

    # 5) Calificaciones de la empresa, dentro del aria-label
    for columna, etiqueta in CALIFICACIONES.items():
        elementos = navegador.find_elements(By.XPATH, f"//*[starts-with(@aria-label, '{etiqueta} ')]")
        if elementos:
            numero = re.search(r"\d+(\.\d+)?", elementos[0].get_attribute("aria-label"))
            aviso[columna] = float(numero.group())

    interesados = navegador.find_elements(By.XPATH, "//*[contains(@aria-label, 'personas quieren trabajar aquí')]")
    if interesados:
        cantidad = interesados[0].get_attribute("aria-label").split()[0]
        aviso["personas_quieren_trabajar_aqui"] = int(cantidad.replace(".", "").replace(",", ""))

    return aviso

Se prueba con el aviso de ejemplo antes de lanzarla sobre todos:

In [18]:
ejemplo = leer_aviso(navegador, url_ejemplo)
pd.Series(ejemplo)

url                               https://www.bumeran.com.pe/empleos/analista-de-datos-fra...
antiguedad_texto                                                    Actualizado hace 12 horas
area                                                Tecnología, Sistemas y Telecomunicaciones
personas_quieren_trabajar_aqui                                                           None
modalidad                                                                             Híbrido
subarea                                                                 Tecnologia / Sistemas
jornada_y_contrato                                                        Full-time, Temporal
nivel_laboral                                                                         Semi Sr
vacantes_texto                                                         2 vacantes disponibles
apto_discapacidad_texto                                                                  None
salario_texto                                               

### 3.5 Recorrer todos los avisos

Aquí se va la mayor parte del tiempo. Dos cuidados:

- Si un aviso **no carga** (lo retiraron, o tardó demasiado), se anota y se sigue.
- Si Bumeran **bloquea**, se detiene todo: insistir solo alarga el bloqueo.
  Lo leído hasta ese momento se conserva.

In [19]:
urls = avisos["url"].tolist()
if LIMITE_AVISOS:
    urls = urls[:LIMITE_AVISOS]

detalles = []
fallidos = []
inicio = time.time()

for numero, url in enumerate(urls, start=1):
    try:
        detalles.append(leer_aviso(navegador, url))
    except (TimeoutException, StaleElementReferenceException):
        if "you have been blocked" in navegador.page_source:
            print("Bumeran bloqueó el acceso: se detiene aquí.")
            break
        fallidos.append(url)

    if numero % 25 == 0:
        print(f"  {numero}/{len(urls)} avisos · {(time.time() - inicio) / 60:.1f} min")
    time.sleep(PAUSA_SEGUNDOS)

print(f"\nLeídos: {len(detalles)} · Fallidos: {len(fallidos)} · {(time.time() - inicio) / 60:.1f} minutos")

  25/288 avisos · 2.0 min


  50/288 avisos · 4.0 min


  75/288 avisos · 6.1 min


  100/288 avisos · 8.3 min


  125/288 avisos · 10.3 min


...

  200/288 avisos · 16.6 min


  225/288 avisos · 18.6 min


  250/288 avisos · 20.7 min


  275/288 avisos · 22.8 min



Leídos: 288 · Fallidos: 0 · 23.9 minutos


In [20]:
# Segundo intento para los que fallaron: a veces la página solo tardó más
pendientes = fallidos
fallidos = []
for url in pendientes:
    try:
        detalles.append(leer_aviso(navegador, url))
    except (TimeoutException, StaleElementReferenceException):
        fallidos.append(url)
    time.sleep(PAUSA_SEGUNDOS)

print("Leídos:", len(detalles), "· Siguen fallando:", len(fallidos))
fallidos

Leídos: 288 · Siguen fallando: 0


[]

In [21]:
navegador.quit()      # ya no hace falta el navegador: de aquí en adelante es pandas

## 4. Etapa 3 · Limpiar y enriquecer

In [22]:
detalle = pd.DataFrame(detalles)
empleos = avisos.merge(detalle, on="url", how="inner")
empleos.shape

(288, 42)

### Identificador y fechas

El número del aviso se guarda como **texto**: es un código, no una cantidad
(igual que el ubigeo).

El JSON-LD trae las fechas en hora universal: `2026-09-18T05:00:00Z` es la
medianoche del 18 en Lima, que está 5 horas atrás. Se pasan a hora de Lima
antes de quedarse con el día; si no, algunas fechas se correrían un día.

Para los avisos sin bloque `JobPosting` se usa la fecha visible («Publicado el
18/09/2026»), que es la misma.

In [23]:
empleos["id_aviso"] = empleos["url"].str.extract(r"-(\d+)\.html$")[0]

publicacion = pd.to_datetime(empleos["fecha_publicacion_iso"], utc=True).dt.tz_convert("America/Lima")
vencimiento = pd.to_datetime(empleos["fecha_vencimiento_iso"], utc=True).dt.tz_convert("America/Lima")
publicado_visible = pd.to_datetime(
    empleos["publicado_el_texto"].str.replace("Publicado el ", ""), format="%d/%m/%Y", errors="coerce"
)

empleos["fecha_publicacion"] = pd.to_datetime(publicacion.dt.date).fillna(publicado_visible)
empleos["fecha_vencimiento"] = pd.to_datetime(vencimiento.dt.date)
empleos["dias_publicado"] = (pd.Timestamp(FECHA_EXTRACCION.date()) - empleos["fecha_publicacion"]).dt.days
empleos["fecha_extraccion"] = FECHA_EXTRACCION.strftime("%Y-%m-%d %H:%M")

print("Avisos sin fecha:", empleos["fecha_publicacion"].isna().sum())
empleos[["id_aviso", "fecha_publicacion_iso", "publicado_el_texto", "fecha_publicacion", "dias_publicado"]].head()

Avisos sin fecha: 0


,id_aviso,fecha_publicacion_iso,publicado_el_texto,fecha_publicacion,dias_publicado
0,1118448331,2026-09-18T05:00:00.000Z,Publicado el 18/09/2026,2026-09-18,5
1,1118404352,2026-08-17T05:00:00.000Z,Publicado el 17/08/2026,2026-08-17,37
2,1118369996,2026-07-21T05:00:00.000Z,Publicado el 21/07/2026,2026-07-21,64
3,1118398943,2026-08-11T05:00:00.000Z,Publicado el 11/08/2026,2026-08-11,43
4,1118386707,2026-07-31T05:00:00.000Z,Publicado el 31/07/2026,2026-07-31,54


### Números y marcas

- «Full-time, Indeterminado» son dos datos: jornada y tipo de contrato.
- «2 vacantes disponibles» pasa a ser el número 2.
- «De S/.1,800 A S/.2,100 Por Mes» se parte en mínimo, máximo, moneda y periodo.
- Un mismo aviso a veces se publica dos veces con distinto número. Se marca
  la segunda copia (mismo puesto, empresa y descripción) en vez de borrarla.

In [24]:
empleos["jornada"] = empleos["jornada_y_contrato"].str.split(",").str[0].str.strip()
empleos["tipo_contrato"] = empleos["jornada_y_contrato"].str.split(",").str[1].str.strip()

empleos["vacantes"] = empleos["vacantes_texto"].str.extract(r"(\d+)")[0].astype("Int64")
empleos["apto_discapacidad"] = empleos["apto_discapacidad_texto"].notna()
empleos["empresa_confidencial"] = empleos["empresa"] == "Confidencial"
empleos["largo_descripcion"] = empleos["descripcion"].str.len()
empleos["aviso_repetido"] = empleos.duplicated(subset=["puesto", "empresa", "descripcion"], keep="first")

empleos[["jornada_y_contrato", "jornada", "tipo_contrato", "vacantes_texto", "vacantes"]].head()

,jornada_y_contrato,jornada,tipo_contrato,vacantes_texto,vacantes
0,"Full-time, Temporal",Full-time,Temporal,1 vacante disponible,1
1,"Full-time, Indeterminado",Full-time,Indeterminado,1 vacante disponible,1
2,"Full-time, Indeterminado",Full-time,Indeterminado,1 vacante disponible,1
3,"Full-time, Indeterminado",Full-time,Indeterminado,1 vacante disponible,1
4,"Full-time, Temporal",Full-time,Temporal,1 vacante disponible,1


In [25]:
montos = empleos["salario_texto"].str.findall(r"\d[\d,]*")      # ["1,800", "2,100"]
empleos["salario_min"] = montos.str[0].str.replace(",", "").astype(float)
empleos["salario_max"] = montos.str[-1].str.replace(",", "").astype(float)
empleos["salario_periodo"] = empleos["salario_texto"].str.extract(r"Por (\w+)")[0]

empleos["salario_moneda"] = None
empleos.loc[empleos["salario_texto"].str.contains("S/", na=False, regex=False), "salario_moneda"] = "PEN"
empleos.loc[empleos["salario_texto"].str.contains("US$", na=False, regex=False), "salario_moneda"] = "USD"

empleos.loc[empleos["salario_texto"].notna(), ["puesto", "salario_texto", "salario_min", "salario_max", "salario_moneda"]].head()

,puesto,salario_texto,salario_min,salario_max,salario_moneda
14,Analista de BI y Automatización (Especialista),"De S/.3,500 A S/.4,000 Por Mes",3500.0,4000.0,PEN
40,Analista de datos - Analista BI,"De S/.2,000 A S/.2,500 Por Mes",2000.0,2500.0,PEN
44,Analista de Datos / Business Intelligence,"De S/.1,800 A S/.2,100 Por Mes",1800.0,2100.0,PEN
45,Analista de Datos / Business Intelligence,"De S/.1,800 A S/.2,100 Por Mes",1800.0,2100.0,PEN
46,Analista de Datos,"De S/.1,800 A S/.1,850 Por Mes",1800.0,1850.0,PEN


### Ubicación

Cada empresa escribe el lugar a su manera. Se unifican los nombres que se
repitieron con distinta forma («Gobierno Regional del Callao», «Surco»,
«Distrito de San Luis»). Para los avisos sin bloque `JobPosting`, el distrito
y el departamento salen del texto visible: «Miraflores, Lima, Peru».

In [26]:
partes = empleos["ubicacion_texto"].str.split(", ")
empleos["distrito"] = empleos["distrito"].fillna(partes.str[-3])
empleos["departamento"] = empleos["departamento"].fillna(partes.str[-2])

NOMBRES_DE_LUGAR = {
    "Gobierno Regional del Callao": "Callao",
    "Cuzco": "Cusco",
    "Surco": "Santiago de Surco",
    "Magdalena": "Magdalena del Mar",
    "Ate Vitarte": "Ate",
}
empleos["distrito"] = empleos["distrito"].str.replace("Distrito de ", "", regex=False)
empleos["distrito"] = empleos["distrito"].replace(NOMBRES_DE_LUGAR)
empleos["departamento"] = empleos["departamento"].replace(NOMBRES_DE_LUGAR)

empleos["departamento"].value_counts(dropna=False)

departamento
Lima           262
La Libertad      9
Arequipa         8
Callao           3
Ica              2
Apurimac         1
Cusco            1
Pasco            1
San Martín       1
Name: count, dtype: int64

### ¿Qué herramientas pide cada aviso?

Se busca cada herramienta en el título y en la descripción. Al revisar los
avisos aparecieron tres trampas:

- «Phyton» y «Tableu» vienen mal escritos en varios avisos: se cuentan igual.
- Buscar `excel` a secas cuenta «excelente». `\b` exige que sea la palabra completa.
- **R** es una sola letra: solo cuenta en mayúscula y entre separadores
  («Python, R, SQL»). Es la única búsqueda que distingue mayúsculas.

In [27]:
HERRAMIENTAS = {
    "SQL": r"\bsql\b",
    "Python": r"python|phyton",
    "R": r"[\s,(/]R[\s,)/;.]|RStudio",
    "Power BI": r"power\s?bi",
    "Excel": r"\bexcel\b",
    "Tableau": r"tableau|tableu",
    "Qlik": r"qlik",
    "Looker": r"looker",
    "SAS": r"\bsas\b",
    "SPSS": r"spss",
    "Stata": r"\bstata\b",
    "Spark": r"spark",
    "Databricks": r"databricks",
    "AWS": r"\baws\b|amazon web services",
    "Azure": r"azure",
    "Google Cloud": r"\bgcp\b|google cloud|bigquery",
    "SAP": r"\bsap\b",
    "Snowflake": r"snowflake",
    "ETL": r"\betl\b",
    "Machine Learning": r"machine learning|aprendizaje automático",
}

texto_aviso = empleos["puesto"].fillna("") + "\n" + empleos["descripcion"].fillna("")

columnas_herramientas = {}
for nombre, patron in HERRAMIENTAS.items():
    columna = "pide_" + nombre.lower().replace(" ", "_")
    columnas_herramientas[nombre] = columna
    # case=True (distinguir mayúsculas) solo para R
    empleos[columna] = texto_aviso.str.contains(patron, case=(nombre == "R"), regex=True)

list(columnas_herramientas.values())

['pide_sql',
 'pide_python',
 'pide_r',
 'pide_power_bi',
 'pide_excel',
 'pide_tableau',
 'pide_qlik',
 'pide_looker',
 'pide_sas',
 'pide_spss',
 'pide_stata',
 'pide_spark',
 'pide_databricks',
 'pide_aws',
 'pide_azure',
 'pide_google_cloud',
 'pide_sap',
 'pide_snowflake',
 'pide_etl',
 'pide_machine_learning']

In [28]:
# Una columna legible con todas las herramientas del aviso: "SQL, Python, Power BI"
empleos["herramientas"] = ""
for nombre, columna in columnas_herramientas.items():
    empleos.loc[empleos[columna], "herramientas"] += nombre + ", "
empleos["herramientas"] = empleos["herramientas"].str.rstrip(", ")

empleos["n_herramientas"] = empleos[list(columnas_herramientas.values())].sum(axis=1)
empleos[["puesto", "herramientas", "n_herramientas"]].head(8)

,puesto,herramientas,n_herramientas
0,430 Analista de Datos,"Power BI, Databricks, AWS",3
1,Analista BI - Business Intelligence,"SQL, Python, Power BI, ETL",4
2,Analista BI Comercial,"SQL, Power BI, Excel",3
3,ANALISTA BI COMERCIAL- Rubro Gastronómico,"SQL, Power BI, Excel",3
4,Analista BI Jr,"SQL, Power BI, Tableau, Snowflake",4
5,Analista Contable (Business Intelligence),"SQL, Python, Power BI, Excel, SAP",5
6,Analista de Administración de Personal - Power Automate ...,"Power BI, SAP",2
7,Analista de Administrativo/ Manejo de Bases de Datos/ Se...,Excel,1


### Años de experiencia (aproximado)

Bumeran no tiene un campo de experiencia: está escrita dentro de la
descripción, cada empresa a su manera. La regla es simple:

1. Se parte la descripción en frases (punto seguido, punto y coma, salto de línea).
2. Se miran solo las frases que dicen «experiencia» y «año».
3. Se descartan las frases en que la empresa habla de sí misma: «somos una
   empresa con más de 30 años de experiencia **en el mercado**».
4. De los números que quedan (hasta 15), se toma el menor: «de 2 a 3 años» → 2,
   «1.5 a 3 años» → 1.5.

Es una aproximación. Sirve para comparar grupos, no para leer un aviso suelto.

In [29]:
PALABRAS_DE_LA_EMPRESA = ["mercado", "trayectoria", "somos", "contamos", "fundad", "líder", "nuestra empresa"]


def anios_de_experiencia(descripcion):
    if not isinstance(descripcion, str):
        return None
    candidatos = []
    # Se corta en punto seguido de espacio, no en cualquier punto: "1.5 años" debe quedar entero
    for frase in re.split(r"\n|;|\.\s", descripcion.lower()):
        if "experiencia" not in frase or "año" not in frase:
            continue
        if any(palabra in frase for palabra in PALABRAS_DE_LA_EMPRESA):
            continue
        for numero in re.findall(r"\d+\.?\d*", frase):
            if float(numero) <= 15:
                candidatos.append(float(numero))
    if candidatos:
        return min(candidatos)
    return None


empleos["experiencia_min_anios"] = empleos["descripcion"].apply(anios_de_experiencia)
print("Avisos con años de experiencia detectados:", empleos["experiencia_min_anios"].notna().sum(), "de", len(empleos))

Avisos con años de experiencia detectados: 216 de 288


**Control de calidad.** Si la regla funciona, los puestos Junior deberían pedir
menos años que los Senior:

In [30]:
empleos.groupby("nivel_laboral")["experiencia_min_anios"].agg(["count", "median", "min", "max"])

,count,median,min,max
nivel_laboral,,,,
Gerencia / Alta Gerencia / Dirección,1,5.0,5.0,5.0
Jefe / Supervisor / Responsable,2,3.5,3.0,4.0
Junior,59,2.0,1.0,5.0
No especificado,3,2.0,1.0,2.0
Otro,0,NaN,NaN,NaN
Semi Sr,105,2.0,1.0,5.0
Senior,41,3.0,1.0,6.0
Senior / Semi-Senior,4,3.0,2.0,3.0
Sin Experiencia,1,2.0,2.0,2.0


### Otras variables

- `pide_ingles`: la descripción menciona inglés.
- `familia`: si el aviso salió en alguna búsqueda del perfil analista, es
  «Análisis de datos / BI»; si salió solo en las afines, «Ciencia e
  ingeniería de datos».

Al revisar los títulos apareció un intruso: la búsqueda *analista de
informacion* también trae avisos de **seguridad de la información**
(ciberseguridad), que no son puestos de datos. Van en una tercera familia,
para poder excluirlos.

In [31]:
empleos["pide_ingles"] = texto_aviso.str.lower().str.contains(r"ingl[eé]s|english", regex=True)



def familia_del_aviso(terminos_busqueda):
    for termino in terminos_busqueda.split("; "):
        if termino in TERMINOS_ANALISTA:
            return "Análisis de datos / BI"
    return "Ciencia e ingeniería de datos"


empleos["familia"] = empleos["terminos_busqueda"].apply(familia_del_aviso)

es_seguridad = empleos["puesto"].str.lower().str.contains("seguridad")
empleos.loc[es_seguridad, "familia"] = "Seguridad de la información"

empleos["familia"].value_counts()

familia
Análisis de datos / BI           185
Ciencia e ingeniería de datos     84
Seguridad de la información       19
Name: count, dtype: int64

In [32]:
empleos.loc[es_seguridad, "puesto"].head()

23     Analista de CLOUD Seguridad de la Información y Ciberseg...
121               Analista de Seguridad de Información | IAM / PAM
122                        Analista de Seguridad de la Información
123                  Analista de Seguridad de la Información - GRC
124                        Analista de Seguridad de la Información
Name: puesto, dtype: object

### Ordenar las columnas

De lo general a lo particular: qué puesto es, dónde y cuándo, qué pide, quién
lo ofrece, y al final el texto completo.

In [33]:
# Excel rechaza algunos caracteres de control invisibles que llegan pegados desde Word
for columna in ["puesto", "descripcion", "empresa_descripcion"]:
    empleos[columna] = empleos[columna].str.replace(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", regex=True)

COLUMNAS = (
    ["id_aviso", "puesto", "empresa", "familia", "terminos_busqueda", "n_busquedas",
     "area", "subarea", "nivel_laboral", "modalidad", "jornada", "tipo_contrato",
     "tipo_empleo_schema", "vacantes", "multiples_vacantes", "apto_discapacidad",
     "postulacion_rapida", "alta_revision_perfiles", "aviso_nuevo",
     "departamento", "distrito", "pais", "ubicacion_texto",
     "fecha_publicacion", "fecha_vencimiento", "dias_publicado", "antiguedad_texto",
     "salario_texto", "salario_min", "salario_max", "salario_moneda", "salario_periodo",
     "experiencia_min_anios", "pide_ingles", "herramientas", "n_herramientas"]
    + list(columnas_herramientas.values())
    + ["empresa_confidencial", "empresa_verificada", "empresa_id", "empresa_url",
       "empresa_industria", "empresa_ubicacion", "empresa_tamano", "empresa_descripcion"]
    + list(CALIFICACIONES)
    + ["personas_quieren_trabajar_aqui",
       "descripcion", "largo_descripcion", "aviso_repetido", "url", "fecha_extraccion"]
)

# Columnas intermedias que ya se transformaron en otras
AUXILIARES = ["fecha_publicacion_iso", "fecha_vencimiento_iso", "publicado_el_texto", "jornada_y_contrato",
              "vacantes_texto", "apto_discapacidad_texto"]

no_previstas = []
for columna in empleos.columns:
    if columna not in COLUMNAS and columna not in AUXILIARES:
        no_previstas.append(columna)
print("Columnas que aparecieron sin estar previstas:", no_previstas)

empleos = empleos[COLUMNAS + no_previstas]
empleos = empleos.sort_values(["familia", "fecha_publicacion"], ascending=[True, False])
empleos = empleos.reset_index(drop=True)
empleos.shape

Columnas que aparecieron sin estar previstas: []


(288, 77)

Así queda **un aviso** completo:

In [34]:
empleos.iloc[0]

id_aviso                                                                           1118453111
puesto                                                           Analista de Datos Operativos
empresa                                                                                  OSLO
familia                                                                Análisis de datos / BI
terminos_busqueda                                                           analista de datos
n_busquedas                                                                                 1
area                                                Tecnología, Sistemas y Telecomunicaciones
subarea                                                                     Análisis de Datos
nivel_laboral                                                                 No especificado
modalidad                                                                             Híbrido
jornada                                                     

In [35]:
empleos.head()

,id_aviso,puesto,empresa,familia,terminos_busqueda,n_busquedas,area,subarea,nivel_laboral,modalidad,jornada,tipo_contrato,tipo_empleo_schema,vacantes,multiples_vacantes,apto_discapacidad,postulacion_rapida,alta_revision_perfiles,aviso_nuevo,departamento,distrito,pais,ubicacion_texto,fecha_publicacion,fecha_vencimiento,dias_publicado,antiguedad_texto,salario_texto,salario_min,salario_max,salario_moneda,salario_periodo,experiencia_min_anios,pide_ingles,herramientas,n_herramientas,pide_sql,pide_python,pide_r,pide_power_bi,pide_excel,pide_tableau,pide_qlik,pide_looker,pide_sas,pide_spss,pide_stata,pide_spark,pide_databricks,pide_aws,pide_azure,pide_google_cloud,pide_sap,pide_snowflake,pide_etl,pide_machine_learning,empresa_confidencial,empresa_verificada,empresa_id,empresa_url,empresa_industria,empresa_ubicacion,empresa_tamano,empresa_descripcion,calif_general,calif_ambiente_laboral,calif_balance_vida_trabajo,calif_beneficios,calif_desarrollo_profesional,calif_gestion_liderazgo,calif_salario,personas_quieren_trabajar_aqui,descripcion,largo_descripcion,aviso_repetido,url,fecha_extraccion
0,1118453111,Analista de Datos Operativos,OSLO,Análisis de datos / BI,analista de datos,1,"Tecnología, Sistemas y Telecomunicaciones",Análisis de Datos,No especificado,Híbrido,Full-time,Indeterminado,FULL_TIME,1,False,False,False,False,True,Lima,San Borja,PE,"San Borja, Lima, Peru",2026-09-23,2026-12-22,0,Publicado hace 10 horas,None,NaN,NaN,None,NaN,2.0,False,"Power BI, Excel",2,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,13306606,https://www.bumeran.com.pe/perfiles/empresa_oslo_1330660...,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,¡Somos el equipo que le pone punche a la logística! Nos ...,2216,False,https://www.bumeran.com.pe/empleos/analista-de-datos-ope...,2026-09-23 21:32
1,1118453948,Data Analyst,San Fernando - La buena Familia,Análisis de datos / BI,data analyst,1,Oficios y Otros,Otros,Junior,Híbrido,Full-time,Indeterminado,FULL_TIME,1,False,False,False,False,True,Lima,Surquillo,PE,"Surquillo, Lima, Peru",2026-09-23,2026-12-22,0,Publicado hace 5 horas,None,NaN,NaN,None,NaN,1.0,True,"Python, Power BI",2,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,10002829,https://www.bumeran.com.pe/perfiles/empresa_san-fernando...,Consumo masivo,"LIMA, Lima",Entre 1001 y 5000 empleados,Somos una empresa peruana con mpas de 75 años en el merc...,3.8,3.6,3.6,3.9,3.6,3.4,3.3,30.0,"En San Fernando, buscamos un Data Analyst que actúe como...",3754,False,https://www.bumeran.com.pe/empleos/data-analyst-san-fern...,2026-09-23 21:32
2,1118450782,Analista de Información Comercial,Confidencial,Análisis de datos / BI,analista de informacion,1,"Comercial, Ventas y Negocios",Comercial,Semi Sr,Presencial,Full-time,Indeterminado,FULL_TIME,1,False,False,False,True,False,Lima,La Victoria,PE,"La Victoria, Lima, Peru",2026-09-22,2026-12-21,1,Actualizado hace 10 horas,None,NaN,NaN,None,NaN,NaN,False,"Python, Power BI, Excel, SAP",4,False,True,False,True,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,True,False,None,None,None,None,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Puesto: Analista de Información Comercial\nModalidad: Pr...,1040,False,https://www.bumeran.com.pe/empleos/analista-de-informaci...,2026-09-23 21:32
3,1118452239,Data Analyst,FARMACIAS PERUANAS,Análisis de datos / BI,data analyst,1,"Tecnología, Sistemas y Telecomunicaciones",Análisis de Datos,Semi Sr,Híbrido,Full-time,Indeterminado,FULL_TIME,1,False,False,True,False,False,Lima,La Victoria,PE,"La Victoria, Lima, Peru",2026-09-22,2026-12-21,1,Publicado ayer,None,NaN,NaN,None,NaN,1.0,True,"SQL, Power BI, Excel, Google Cloud",4,True,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,1673060,https://www.bumeran.com.pe/perfile

## 5. Etapa 4 · Guardar

Dos archivos con la misma tabla:

- **CSV** en `utf-8-sig`: esa firma al inicio hace que Excel muestre bien las
  tildes y la ñ al abrirlo con doble clic.
- **Excel** con dos hojas: los avisos y el **diccionario de variables**, que
  explica cada columna y de dónde sale.

El nombre lleva la fecha: los avisos cambian todos los días, y si se repite la
extracción cada semana se arma una serie de tiempo.

In [36]:
DESCRIPCIONES = {
    "id_aviso": ("Bumeran", "Número del aviso en Bumeran (texto, es un código)"),
    "puesto": ("JSON-LD", "Título del puesto tal como lo publicó la empresa"),
    "empresa": ("JSON-LD", "Nombre de la empresa; 'Confidencial' si no se revela"),
    "familia": ("Calculada", "'Análisis de datos / BI' o 'Ciencia e ingeniería de datos' (según las búsquedas en que apareció); 'Seguridad de la información' si el título dice 'seguridad'"),
    "terminos_busqueda": ("Búsqueda", "Búsquedas de Bumeran en que apareció el aviso, separadas por ';'"),
    "n_busquedas": ("Búsqueda", "En cuántas búsquedas apareció"),
    "area": ("Breadcrumb", "Área del aviso según Bumeran"),
    "subarea": ("Atributos", "Subárea del aviso según Bumeran"),
    "nivel_laboral": ("Atributos", "Seniority: Junior, Semi Sr, Senior, Jefe, etc."),
    "modalidad": ("Atributos", "Presencial, Híbrido o Remoto"),
    "jornada": ("Atributos", "Full-time, Part-time, etc."),
    "tipo_contrato": ("Atributos", "Indeterminado, Temporal, etc."),
    "tipo_empleo_schema": ("JSON-LD", "Tipo de empleo en el estándar schema.org (FULL_TIME, TEMPORARY...)"),
    "vacantes": ("Atributos", "Número de vacantes"),
    "multiples_vacantes": ("Marca", "True si el aviso lleva la marca 'Múltiples vacantes'"),
    "apto_discapacidad": ("Atributos", "True si el aviso es apto para personas con discapacidad"),
    "postulacion_rapida": ("Marca", "True si permite 'Postulación rápida'"),
    "alta_revision_perfiles": ("Marca", "True si la empresa revisa muchos perfiles ('Alta revisión de perfiles')"),
    "aviso_nuevo": ("Marca", "True si Bumeran lo marca como 'Nuevo'"),
    "departamento": ("JSON-LD / Página", "Departamento del puesto (nombres unificados)"),
    "distrito": ("JSON-LD / Página", "Distrito (o ciudad) del puesto (nombres unificados)"),
    "pais": ("JSON-LD", "País (código ISO)"),
    "ubicacion_texto": ("Página", "Ubicación tal como se muestra: 'Distrito, Departamento, País'"),
    "fecha_publicacion": ("JSON-LD / Página", "Fecha de publicación original (hora de Lima)"),
    "fecha_vencimiento": ("JSON-LD", "Fecha hasta la que el aviso está vigente"),
    "dias_publicado": ("Calculada", "Días entre la publicación y la extracción"),
    "antiguedad_texto": ("Página", "Texto de antigüedad: 'Publicado hace 3 días', 'Actualizado ayer'"),
    "salario_texto": ("Atributos", "Sueldo tal como se publica: 'De S/.1,800 A S/.2,100 Por Mes' (vacío si no se publica)"),
    "salario_min": ("Calculada", "Extremo inferior del rango de sueldo publicado"),
    "salario_max": ("Calculada", "Extremo superior del rango de sueldo publicado"),
    "salario_moneda": ("Calculada", "PEN (soles) o USD"),
    "salario_periodo": ("Calculada", "Periodo del sueldo: Mes, Hora..."),
    "experiencia_min_anios": ("Calculada", "Años mínimos de experiencia detectados en la descripción (aproximado)"),
    "pide_ingles": ("Calculada", "True si la descripción menciona inglés"),
    "herramientas": ("Calculada", "Herramientas mencionadas en título o descripción"),
    "n_herramientas": ("Calculada", "Cantidad de herramientas mencionadas"),
    "empresa_confidencial": ("Calculada", "True si la empresa no se identifica"),
    "empresa_verificada": ("Página", "True si Bumeran marca a la empresa como verificada"),
    "empresa_id": ("JSON-LD", "Código de la empresa en Bumeran"),
    "empresa_url": ("JSON-LD", "Perfil de la empresa en Bumeran"),
    "empresa_industria": ("Ficha empresa", "Industria o sector de la empresa"),
    "empresa_ubicacion": ("Ficha empresa", "Sede de la empresa (puede diferir de la del puesto)"),
    "empresa_tamano": ("Ficha empresa", "Rango de número de empleados"),
    "empresa_descripcion": ("Ficha empresa", "Presentación de la empresa"),
    "calif_general": ("Ficha empresa", "Calificación general de la empresa por sus empleados (1 a 5)"),
    "calif_ambiente_laboral": ("Ficha empresa", "Calificación: ambiente laboral (1 a 5)"),
    "calif_balance_vida_trabajo": ("Ficha empresa", "Calificación: balance entre vida personal y trabajo (1 a 5)"),
    "calif_beneficios": ("Ficha empresa", "Calificación: beneficios (1 a 5)"),
    "calif_desarrollo_profesional": ("Ficha empresa", "Calificación: desarrollo profesional (1 a 5)"),
    "calif_gestion_liderazgo": ("Ficha empresa", "Calificación: gestión y liderazgo (1 a 5)"),
    "calif_salario": ("Ficha empresa", "Calificación: salario (1 a 5)"),
    "personas_quieren_trabajar_aqui": ("Ficha empresa", "Personas que marcaron que quieren trabajar en la empresa"),
    "descripcion": ("Página", "Descripción completa del aviso"),
    "largo_descripcion": ("Calculada", "Número de caracteres de la descripción"),
    "aviso_repetido": ("Calculada", "True si es copia de otro aviso (mismo puesto, empresa y descripción)"),
    "url": ("Bumeran", "Dirección del aviso"),
    "fecha_extraccion": ("Calculada", "Fecha y hora en que se extrajo"),
}
for nombre, columna in columnas_herramientas.items():
    DESCRIPCIONES[columna] = ("Calculada", f"True si el título o la descripción mencionan {nombre}")

filas = []
for columna in empleos.columns:
    fuente, descripcion = DESCRIPCIONES.get(columna, ("?", "(sin descripción)"))
    filas.append({"columna": columna, "fuente": fuente, "descripcion": descripcion,
                  "completos": empleos[columna].notna().sum()})

diccionario = pd.DataFrame(filas)
diccionario

,columna,fuente,descripcion,completos
0,id_aviso,Bumeran,"Número del aviso en Bumeran (texto, es un código)",288
1,puesto,JSON-LD,Título del puesto tal como lo publicó la empresa,288
2,empresa,JSON-LD,Nombre de la empresa; 'Confidencial' si no se revela,288
3,familia,Calculada,'Análisis de datos / BI' o 'Ciencia e ingeniería de dato...,288
4,terminos_busqueda,Búsqueda,"Búsquedas de Bumeran en que apareció el aviso, separadas...",288
5,n_busquedas,Búsqueda,En cuántas búsquedas apareció,288
6,area,Breadcrumb,Área del aviso según Bumeran,288
7,subarea,Atributos,Subárea del aviso según Bumeran,288
8,nivel_laboral,Atributos,"Seniority: Junior, Semi Sr, Senior, Jefe, etc.",288
9,modalidad,Atributos,"Presencial, Híbrido o Remoto",288


In [37]:
empleos.to_csv(ARCHIVO_CSV, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(ARCHIVO_EXCEL) as excel:
    empleos.to_excel(excel, sheet_name="avisos", index=False)
    diccionario.to_excel(excel, sheet_name="diccionario", index=False)

print("Guardados:", ARCHIVO_CSV, "y", ARCHIVO_EXCEL)
print(len(empleos), "avisos ·", len(empleos.columns), "columnas")

Guardados: bumeran_analista_datos_2026-09-23.csv y bumeran_analista_datos_2026-09-23.xlsx
288 avisos · 77 columnas


## 6. Etapa 5 · Primeras respuestas

Con la tabla armada, cada pregunta es una línea de pandas. Salvo que se diga
otra cosa, se cuentan los avisos únicos (sin las copias repetidas).

In [38]:
unicos = empleos[~empleos["aviso_repetido"]]
analistas = unicos[unicos["familia"] == "Análisis de datos / BI"]

print("Avisos únicos:", len(unicos))
unicos["familia"].value_counts()

Avisos únicos: 281


familia
Análisis de datos / BI           181
Ciencia e ingeniería de datos     81
Seguridad de la información       19
Name: count, dtype: int64

### ¿Presencial, híbrido o remoto?

In [39]:
(pd.crosstab(unicos["modalidad"], unicos["familia"], normalize="columns") * 100).round(1)

familia,Análisis de datos / BI,Ciencia e ingeniería de datos,Seguridad de la información
modalidad,,,
Híbrido,37.6,56.8,36.8
Presencial,60.2,35.8,47.4
Remoto,2.2,7.4,15.8


### ¿Qué nivel buscan?

In [40]:
analistas["nivel_laboral"].value_counts()

nivel_laboral
Semi Sr                                 78
Junior                                  58
Senior                                  25
Senior / Semi-Senior                     5
No especificado                          4
Trainee / Pasante                        4
Jefe / Supervisor / Responsable          3
Sin Experiencia                          2
Otro                                     1
Gerencia / Alta Gerencia / Dirección     1
Name: count, dtype: int64

### ¿Dónde están?

In [41]:
unicos["departamento"].value_counts()

departamento
Lima           256
La Libertad      9
Arequipa         8
Callao           2
Ica              2
Apurimac         1
Cusco            1
San Martín       1
Pasco            1
Name: count, dtype: int64

In [42]:
# Dentro de Lima, por distrito
unicos.loc[unicos["departamento"] == "Lima", "distrito"].value_counts().head(10)

distrito
San Isidro           71
Lima                 51
Miraflores           27
Surquillo            17
Santiago de Surco    16
San Borja            11
La Victoria          11
Cercado De Lima      10
Ate                   7
Callao                6
Name: count, dtype: int64

### ¿Qué herramientas piden a un analista de datos?

Porcentaje de avisos de la familia analista que mencionan cada herramienta:

In [43]:
porcentaje = {}
for nombre, columna in columnas_herramientas.items():
    porcentaje[nombre] = analistas[columna].mean() * 100

pd.Series(porcentaje).sort_values(ascending=False).round(1)

Power BI            75.7
SQL                 61.9
Excel               57.5
Python              49.7
ETL                 21.5
Azure               13.3
AWS                 12.7
Google Cloud        11.0
SAP                  9.9
Looker               8.8
Tableau              7.7
Spark                7.2
Machine Learning     5.5
R                    5.5
Databricks           4.4
Qlik                 2.2
Snowflake            2.2
SPSS                 1.1
SAS                  0.6
Stata                0.6
dtype: float64

### ¿Cuánta experiencia y qué contrato?

In [44]:
print("Mediana de años de experiencia (analistas):", analistas["experiencia_min_anios"].median())
print("Piden inglés:", round(analistas["pide_ingles"].mean() * 100, 1), "%")
analistas["tipo_contrato"].value_counts()

Mediana de años de experiencia (analistas): 2.0
Piden inglés: 16.6 %


tipo_contrato
Indeterminado    135
Temporal          28
Autónomo           5
Pasantía           3
Name: count, dtype: int64

### ¿Quién contrata?

In [45]:
unicos.loc[~unicos["empresa_confidencial"], "empresa"].value_counts().head(10)

empresa
INDRA PERU            23
ManpowerGroup Perú    15
CSTI Corp             10
Protiviti Perú         5
Laureate Perú          5
Encora                 5
NTT DATA               4
EXPERIS PERÚ           4
Inetum Perú            4
Prosegur               4
Name: count, dtype: int64

In [46]:
unicos["empresa_industria"].value_counts().head(10)

empresa_industria
Informática / Tecnología    71
Consultoría                 30
Educación                   20
Banca / Financiera          16
Servicios                   11
Salud                       10
Telecomunicaciones          10
Consumo masivo               6
Retail                       6
Otra                         5
Name: count, dtype: int64

### ¿Y el sueldo?

In [47]:
con_sueldo = unicos[unicos["salario_texto"].notna()]
print("Avisos que publican sueldo:", len(con_sueldo), "de", len(unicos))
con_sueldo[["puesto", "empresa", "nivel_laboral", "salario_min", "salario_max", "familia"]].sort_values("salario_min")

Avisos que publican sueldo: 12 de 281


,puesto,empresa,nivel_laboral,salario_min,salario_max,familia
99,Analista de datos,R&E INNOVA SAC,No especificado,300.0,500.0,Análisis de datos / BI
39,Analista de mejora de procesos + Bi – Contact Center (Pe...,Prosegur,Semi Sr,500.0,4900.0,Análisis de datos / BI
201,Practicante Data Engineer (Hibrido),Arla & Asociados,No especificado,800.0,1000.0,Ciencia e ingeniería de datos
72,Analista de Gestión en Tiempo Real y Análisis de Datos,Ampliffica Perú,Semi Sr,1300.0,1500.0,Análisis de datos / BI
84,Analista de Datos,COBEX,No especificado,1800.0,1850.0,Análisis de datos / BI
108,Analista de datos - Analista BI,DELPILZONE,Semi Sr,2000.0,2500.0,Análisis de datos / BI
136,Analista de datos y Business Intelligence,QENETO,No especificado,2000.0,2500.0,Análisis de datos / BI
114,Analista JR de datos y automatización (facturación),Prosegur,Junior,2200.0,2400.0,Análisis de datos / BI
80,Analista de Datos y Planeamiento,VIETTEL PERU SAC,Semi Sr,2500.0,3500.0,Análisis de datos / BI
171,Analista de BI y Automatización (Especialista),ManpowerGroup Perú,Semi Sr,3500.0,4000.0,Análisis de datos / BI


In [48]:
con_sueldo[["salario_min", "salario_max"]].median()

salario_min    2000.0
salario_max    2500.0
dtype: float64

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Abrir Chrome sin ventana, presentándose como Chrome normal | `Options()` + `--headless=new` + `user-agent=...` |
| Revisar qué permite el sitio | `navegador.get(".../robots.txt")` |
| Buscar sin escribir en el buscador | `/empleos-busqueda-<termino>.html?page=N` |
| Esperar a que carguen los resultados | `WebDriverWait(...).until(EC.text_to_be_present_in_element(...))` |
| Encontrar avisos por el patrón de su dirección | `re.search(r"-\d+\.html$", href)` |
| Una fila por aviso, con sus búsquedas | `groupby("url").agg(terminos_busqueda=("termino", "; ".join))` |
| Leer datos estructurados | `script[type='application/ld+json']` + `json.loads` |
| Saber qué dato es cada línea | nombre del ícono: `i.get_attribute("name")` |
| Leer un valor por su etiqueta | XPath `//p[normalize-space()='Industria']/following-sibling::*[1]` |
| Leer calificaciones | `get_attribute("aria-label")` |
| Seguir si un aviso falla, parar si hay bloqueo | `try / except TimeoutException` |
| Fechas en hora de Lima | `pd.to_datetime(..., utc=True).dt.tz_convert("America/Lima")` |
| Datos de respaldo si falta el JSON-LD | `LocalBusiness`, «Publicado el ...», ubicación visible |
| Sueldo publicado | ícono `icon-light-money` + `str.findall(r"\d[\d,]*")` |
| Herramientas pedidas | `str.contains(patron)` por herramienta |
| Guardar para Excel | `to_csv(..., encoding="utf-8-sig")` + `ExcelWriter` con diccionario |

### Límites de esta base

- **Es una foto del día.** Los avisos cambian a diario; la fecha va en el
  nombre del archivo y en la columna `fecha_extraccion`.
- **La búsqueda trae algunos intrusos.** Los avisos de seguridad de la
  información quedaron en su propia `familia`: para analizar solo datos,
  filtrar `familia != "Seguridad de la información"`.
- **Casi nadie publica sueldo.** Las cifras de sueldo describen solo a los
  pocos avisos que lo muestran, no al mercado.
- **Herramientas y experiencia salen del texto.** Son aproximaciones: que un
  aviso no mencione SQL no garantiza que no lo pida.
- **Copias.** Para contar avisos, filtrar `aviso_repetido == False`.

### Para repetirlo

Correr todo el notebook (*Run → Run All Cells*). Para una prueba rápida, poner
`LIMITE_AVISOS = 10` en la configuración.